<a href="https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I read *The State of AI-Driven SEO* (FlyRank, March 2026; the PDF is in my repo under `docs/`). The paper is careful in ways worth saying first: it calls itself observational, leads with direct comparisons, labels the machine-learning pages as exploratory, reports large sample sizes, and tells the reader the health score is partly built from its own inputs. The questions below are the ones I would ask so that I can practise the same level of care on my own work. They are questions, not corrections.

### Finding A: "The Anatomy of Growing Content" (Finding #1)
*What it says:* rising pages tend to be longer, younger, and slightly better positioned than falling pages (for example, about 3.2K vs. 2.3K average words and 184 vs. 230 days of age).

**Question 1: where does the label come from?** "Rising" and "falling" come from the 30-day vs. previous-30-day trend. The profile numbers (impressions, position) come from the 90-day window, which contains those same 30 days. I would ask whether the profile is measured before the trend window or over a window that includes it.

**Question 2: does the comparison carry the claim?** The paper itself names content age as a confounder. A page that only recently began earning impressions can look "rising" by construction, and young pages are also the ones that began earning recently. I would ask whether the word-count and age gaps still show up *within* age bands (for example, only pages 180–365 days old), which would show the gap is not just an age effect. A small detail: the table lists 74.8K and 45.6K pages while the text lists 74,187 and 45,272. They are probably different filters, and I would ask which one the headline uses.

### Finding B: ML appendix, "What Predicts Growth?" (logistic regression, 71% holdout accuracy)
*What it says:* a logistic regression separates growing from declining pages with 71% holdout accuracy, with content age as the strongest negative signal and recent impressions among the strongest positive ones.

**Question 1: do the features come strictly before the label?** If "recent impressions" cover the same last 30 days that define growing vs. declining, the feature contains part of the answer (an overlapping window). I would ask for the timeline: which dates are features, and which are the label.

**Question 2: does the validation design support "predicts"?** The methodology section says an 80/20 split. I would ask whether it is random by page or grouped by brand. Pages from the same brand share a site, a topic mix and a tracking setup, so a random split can reward memorising a brand. A brand-grouped split answers "does it work on a brand it has never seen?".

**Question 3: what is the base rate?** If the two classes are the rising and falling pages of Finding A (74.8K vs. 45.6K), a model that always says "rising" is right about 62% of the time. The 71% is then about 9 points above that, which is real but smaller than the headline suggests. The paper does not print the base rate or an AUC for this model, so I cannot be sure of the classes, and I would ask.

In [8]:
import os, sys, subprocess, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---- Setup: find the repo root so the CSV loads in Colab and locally ----
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yelsebety/Yassin-Flyrank.git"
REPO_DIR = "Yassin-Flyrank"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

# ---- Finding B, base-rate arithmetic (numbers copied from the paper's Finding #1 table) ----
up, down = 74_800, 45_600
majority_rate = up / (up + down)
print(f"Paper's rising/falling pages: {up:,} vs {down:,}")
print(f"Always guessing 'rising' would be right {majority_rate:.1%} of the time -> 71% is about {71 - 100*majority_rate:.1f} points above that")
print("(assumes the logistic regression uses these two classes; the paper does not print its base rate)\n")

# ---- Finding A, echo on my own starter data: does the age / length / position gap show up? ----
raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
raw["declined"] = (raw["trend_direction"] == "down").astype(int)   # proxy label: evaluation only
elig = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90) & raw["trend_direction"].isin(["up", "down"])]
echo = elig.groupby("trend_direction")[["content_age_days", "word_count", "avg_position"]].median().round(1)
echo["n"] = elig["trend_direction"].value_counts()
print("Median profile of 'up' vs 'down' pages in my starter slice (not the paper's data):")
display(echo)

# Does the age gap survive inside one age band? (the within-band check I asked about in Question 2)
band = elig[(elig["content_age_days"] >= 180) & (elig["content_age_days"] < 365)]
within = band.groupby("trend_direction")[["content_age_days", "word_count"]].median().round(1)
within["n"] = band["trend_direction"].value_counts()
print("\nSame comparison inside ONE age band (180-364 days old):")
display(within)

Paper's rising/falling pages: 74,800 vs 45,600
Always guessing 'rising' would be right 62.1% of the time -> 71% is about 8.9 points above that
(assumes the logistic regression uses these two classes; the paper does not print its base rate)

Median profile of 'up' vs 'down' pages in my starter slice (not the paper's data):


,content_age_days,word_count,avg_position,n
trend_direction,,,,
down,216.0,2909.0,11.3,16262
up,291.5,2847.5,15.3,4388



Same comparison inside ONE age band (180-364 days old):


,content_age_days,word_count,n
trend_direction,,,
down,284.0,3006.5,6028
up,292.0,2735.5,1505


## 2. My model under an honest split (before/after)

My Week-5 model scores pages by how likely they are to be marked "declining". Here I run the **same models on the same pages** under two designs:

- **Before (random, by page):** pages from every client land in both the training and the test part. The model can learn each client's habits and look better than it is.
- **After (grouped by client):** whole clients are held out, so the test asks the honest question: does it work on a client it has never seen?

The gap between the two numbers measures how much of the first score was memorisation. (The starter CSV is one snapshot with no dates, so a time-aware split is not possible. See section 4.)

In [9]:
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEED = 0
# Same slice and features as my Week-4 / Week-5 notebooks
g = raw[(raw["impressions_90d"] >= 500) & (raw["avg_position"] > 0) & (raw["avg_position"] <= 20)
        & (raw["content_age_days"] >= 90)].copy().reset_index(drop=True)
g["log_impressions"]   = np.log1p(g["impressions_90d"])
g["log_ctr"]           = np.log1p(g["ctr"])
g["log_word_count"]    = np.log1p(g["word_count"])
g["log_search_volume"] = np.log1p(g["search_volume"])
NUM = ["log_impressions", "avg_position", "log_ctr", "content_age_days", "days_since_last_update",
       "days_with_impressions", "log_word_count", "log_search_volume", "competition", "cpc", "engagement_rate"]
CAT = ["content_type", "main_intent"]
print(f"Pages: {len(g):,} | clients: {g['client_id'].nunique()} | base rate of the proxy label: {g['declined'].mean():.3f}")

def precision_at_k(y, score, impressions, k=50):
    order = np.lexsort((-np.asarray(impressions), -np.asarray(score)))
    return float(np.asarray(y)[order][:k].mean())

def make_model(name, num_cols):
    num = [("fill", SimpleImputer(strategy="median", add_indicator=True))] + ([("scale", StandardScaler())] if name == "Logistic Regression" else [])
    cat = [("fill", SimpleImputer(strategy="constant", fill_value="unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]
    pre = ColumnTransformer([("num", Pipeline(num), num_cols), ("cat", Pipeline(cat), CAT)])
    est = (LogisticRegression(C=1.0, max_iter=2000) if name == "Logistic Regression"
           else RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED))
    return Pipeline([("pre", pre), ("m", est)])

def cv_scores(df, name, num_cols, splitter, groups=None):
    """Out-of-fold scores per fold: AUC, precision@50, base rate. Metrics are never in-sample."""
    rows, oof = [], pd.Series(np.nan, index=df.index)
    for tr, va in splitter.split(df, df["declined"], groups):
        a, b = df.iloc[tr], df.iloc[va]
        m = make_model(name, num_cols).fit(a[num_cols + CAT], a["declined"])
        p = m.predict_proba(b[num_cols + CAT])[:, 1]
        oof.iloc[va] = p
        rows.append({"auc": roc_auc_score(b["declined"], p), "p@50": precision_at_k(b["declined"], p, b["impressions_90d"]),
                     "base_rate": b["declined"].mean()})
    return pd.DataFrame(rows), oof

random_split  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grouped_split = GroupKFold(n_splits=5)

results, oof_grouped = [], {}
for name in ("Logistic Regression", "Random Forest"):
    r, _  = cv_scores(g, name, NUM, random_split)
    c, oo = cv_scores(g, name, NUM, grouped_split, groups=g["client_id"])
    oof_grouped[name] = oo
    results.append({"model": name, "design": "BEFORE: random by page", "AUC": r["auc"].mean(), "P@50": r["p@50"].mean(), "base rate": r["base_rate"].mean()})
    results.append({"model": name, "design": "AFTER: grouped by client", "AUC": c["auc"].mean(), "P@50": c["p@50"].mean(), "base rate": c["base_rate"].mean()})
before_after = pd.DataFrame(results).round(3)
display(before_after)

pivot = before_after.pivot(index="model", columns="design", values="AUC")
pivot["AUC drop (random -> grouped)"] = pivot["BEFORE: random by page"] - pivot["AFTER: grouped by client"]
print("How much of the random-split AUC disappears when whole clients are held out:")
display(pivot.round(3))

Pages: 12,023 | clients: 28 | base rate of the proxy label: 0.599


,model,design,AUC,P@50,base rate
0,Logistic Regression,BEFORE: random by page,0.666,0.856,0.599
1,Logistic Regression,AFTER: grouped by client,0.608,0.824,0.611
2,Random Forest,BEFORE: random by page,0.709,0.908,0.599
3,Random Forest,AFTER: grouped by client,0.616,0.872,0.611


How much of the random-split AUC disappears when whole clients are held out:


design,AFTER: grouped by client,BEFORE: random by page,AUC drop (random -> grouped)
model,,,
Logistic Regression,0.608,0.666,0.058
Random Forest,0.616,0.709,0.093


**How to read the table:** the BEFORE rows are what a careless random split would have reported. The AFTER rows are the numbers I trust. The "AUC drop" column is the memorisation gap. If it is not zero, the first split was flattering the model. Every row also shows the base rate next to it, because a score only means something relative to it.

## 3. Leakage audit

The same hunt as in Week 3, on my final feature set, using the three kinds of leakage: label-derived features, overlapping windows, and decision-derived features (product flags). The label here is a *proxy*, "down" = impressions in the last 30 days fell more than 20% below the previous 30. It is built from `impressions_last_30d` and `impressions_prev_30d`.

**Checklist**
- Timeline: the features come from the 90-day window, and the label window (last 30 vs. previous 30) sits *inside* it. **This is a real overlap, so I test how much it matters (3b).**
- Label-derived columns: none are used as features (checked in code).
- Product flags: none are in the starter data.
- Population selection: my slice keeps pages with 500+ impressions over the 90 days, a gate that includes the label window. That is a choice I disclose, not a hidden step.
- Split: grouped by client (section 2).
- Base rate: printed next to every metric.
- Top feature: sanity-checked below (3c).

In [10]:
# 3a. Banned columns must not be features (a hard check, not a promise)
BANNED = ["trend_direction", "trend_pct", "declined", "impressions_last_30d", "impressions_prev_30d",
          "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d", "content_id", "client_id"]
assert not set(NUM + CAT) & set(BANNED), "a banned column is a feature"
print("3a. No label-derived, trend or last/prev-30-day columns are features. IDs are used for grouping only.\n")

# 3b. Deliberate leak: add ONE label-derived column and watch the score jump. If it does not, the harness is broken.
NUM_LEAK = NUM + ["trend_pct"]                      # trend_pct is computed from the very windows the label uses
leak_rows = []
for name in ("Logistic Regression", "Random Forest"):
    honest, _ = cv_scores(g, name, NUM, grouped_split, groups=g["client_id"])
    leaky, _  = cv_scores(g, name, NUM_LEAK, grouped_split, groups=g["client_id"])
    leak_rows.append({"model": name, "AUC (honest)": honest["auc"].mean(), "AUC (with trend_pct)": leaky["auc"].mean(),
                      "P@50 (honest)": honest["p@50"].mean(), "P@50 (with trend_pct)": leaky["p@50"].mean()})
leak_table = pd.DataFrame(leak_rows).round(3)
print("3b. Deliberate leak (trend_pct added, then removed - the honest rows are the ones I keep):")
display(leak_table)

3a. No label-derived, trend or last/prev-30-day columns are features. IDs are used for grouping only.

3b. Deliberate leak (trend_pct added, then removed - the honest rows are the ones I keep):


,model,AUC (honest),AUC (with trend_pct),P@50 (honest),P@50 (with trend_pct)
0,Logistic Regression,0.608,0.999,0.824,1.0
1,Random Forest,0.616,1.000,0.872,1.0


In [11]:
# 3c. Overlapping-window test.
# Assumption (check it against the data dictionary): the 90-day window = first 30 days + previous 30 + last 30.
# Then impressions in the first 30 days = 90-day total minus the two label-window pieces: strictly BEFORE the label window.
g["impr_first30"] = g["impressions_90d"] - g["impressions_last_30d"] - g["impressions_prev_30d"]
g["clk_first30"]  = g["clicks_90d"]      - g["clicks_last_30d"]      - g["clicks_prev_30d"]
print(f"Rows where first-30-day impressions come out negative (would break the assumption): {(g['impr_first30'] < 0).sum()}")
print(f"Rows with 0 impressions in the first 30 days (page new to search): {(g['impr_first30'] == 0).mean():.1%}")

g["log_impr_first30"] = np.log1p(g["impr_first30"].clip(lower=0))
g["log_ctr_first30"]  = np.log1p(np.where(g["impr_first30"] > 0, 100 * g["clk_first30"] / g["impr_first30"], np.nan))
NUM_EARLY = ["log_impr_first30", "log_ctr_first30", "content_age_days", "days_since_last_update",
             "log_word_count", "log_search_volume", "competition", "cpc"]

win_rows = []
for name in ("Logistic Regression", "Random Forest"):
    full,  _ = cv_scores(g, name, NUM,       grouped_split, groups=g["client_id"])
    early, _ = cv_scores(g, name, NUM_EARLY, grouped_split, groups=g["client_id"])
    win_rows.append({"model": name, "AUC (90-day features)": full["auc"].mean(), "AUC (first-30-day features only)": early["auc"].mean(),
                     "P@50 (90-day)": full["p@50"].mean(), "P@50 (first-30-day)": early["p@50"].mean(), "base rate": full["base_rate"].mean()})
window_table = pd.DataFrame(win_rows).round(3)
print("\n3c. Same grouped folds. Features from the first 30 days only (before the label window) vs. all 90 days:")
display(window_table)
print("If the first-30-day AUC is clearly lower, part of the 90-day score came from the label window overlapping the features.")

Rows where first-30-day impressions come out negative (would break the assumption): 0
Rows with 0 impressions in the first 30 days (page new to search): 1.0%

3c. Same grouped folds. Features from the first 30 days only (before the label window) vs. all 90 days:


,model,AUC (90-day features),AUC (first-30-day features only),P@50 (90-day),P@50 (first-30-day),base rate
0,Logistic Regression,0.608,0.593,0.824,0.700,0.611
1,Random Forest,0.616,0.582,0.872,0.744,0.611


If the first-30-day AUC is clearly lower, part of the 90-day score came from the label window overlapping the features.


In [12]:
# 3d. Top-feature sanity check: is days_with_impressions a real signal or a 'new to search' artefact?
d = g.copy()
d["days_visible_bucket"] = pd.cut(d["days_with_impressions"], [0, 30, 60, 80, 91], right=False)
bucket = d.groupby("days_visible_bucket", observed=True)["declined"].agg(n="count", decline_rate="mean").round(3)
print("3d. Decline rate by number of days with impressions (bucket table, n printed):")
display(bucket)
print("Few active days = the page only recently began getting impressions. Such pages can look 'up' by construction,")
print("so a model can lean on this without learning real decline risk. I cannot separate those two with this data.")

3d. Decline rate by number of days with impressions (bucket table, n printed):


,n,decline_rate
days_visible_bucket,,
"[0, 30)",101,0.079
"[30, 60)",180,0.606
"[60, 80)",1004,0.738
"[80, 91)",10738,0.590


Few active days = the page only recently began getting impressions. Such pages can look 'up' by construction,
so a model can lean on this without learning real decline risk. I cannot separate those two with this data.


In [13]:
# 3e. Real failure examples (out-of-fold, grouped by client, Random Forest)
g["p_declining"] = oof_grouped["Random Forest"]
cols = ["p_declining", "impressions_90d", "avg_position", "ctr", "days_with_impressions", "content_age_days",
        "content_type", "trend_direction"]
order = np.lexsort((-g["impressions_90d"].to_numpy(), -g["p_declining"].to_numpy()))
ranked = g.iloc[order]
fp = ranked[ranked["declined"] == 0].head(3)[cols]                         # confidently predicted 'down', but were not
fn = g[g["declined"] == 1].sort_values("p_declining").head(3)[cols]        # model gave the lowest risk, but they declined
print("Highest-ranked pages that were NOT 'down' (false alarms):")
display(fp.round(3))
print("Pages that WERE 'down' that the model scored lowest (missed):")
display(fn.round(3))

Highest-ranked pages that were NOT 'down' (false alarms):


,p_declining,impressions_90d,avg_position,ctr,days_with_impressions,content_age_days,content_type,trend_direction
9791,0.904,840,3.1,0.00,85,144,keyword article,stable
2821,0.897,596,16.9,0.00,81,111,keyword article,up
2856,0.885,2032,2.1,0.15,87,144,keyword article,stable


Pages that WERE 'down' that the model scored lowest (missed):


,p_declining,impressions_90d,avg_position,ctr,days_with_impressions,content_age_days,content_type,trend_direction
5027,0.230,148515,5.0,0.81,88,224,keyword article,down
7102,0.240,886,16.3,0.23,24,280,keyword article,down
7563,0.252,88328,6.2,0.70,88,224,keyword article,down


**What to look for in the examples (write your own reading after you run it):** are the false alarms low-CTR pages sitting far down the results page, where a low CTR is common? Are the misses pages with a small drop just past the 20% line (a 21% drop is "down", a 19% drop is "stable")? Is a particular content type or client behind several of them? The honest reading of a failure is a *reason*, not a number.

## 4. Claim rewrite

I take the boldest sentences from my own earlier notebooks and rewrite them so they go no further than the evidence. The rewrite uses these words on purpose: **observed** (what the data shows), **measured** (a number with its comparison), **directional** (a gap too small to be sure of), and **decision-support** (a tool to prioritise human review, not a forecast).

**Limits that apply to everything here:**
- The label is a *proxy* from the last 30 vs. the previous 30 days, inside the same 90-day window as most features (section 3c tests how much that matters).
- The starter CSV has no dates, so I could not use a time-aware split.
- Results are associations in one snapshot; none of them shows that a change to a page would cause a recovery.

In [14]:
# The rewritten claims are built from the numbers computed above, so the sentences cannot drift from the evidence.
rf_after  = before_after.query("model == 'Random Forest' and design.str.startswith('AFTER')", engine="python").iloc[0]
rf_before = before_after.query("model == 'Random Forest' and design.str.startswith('BEFORE')", engine="python").iloc[0]
lr_after  = before_after.query("model == 'Logistic Regression' and design.str.startswith('AFTER')", engine="python").iloc[0]
rf_early  = window_table.query("model == 'Random Forest'").iloc[0]

claims = pd.DataFrame([
    {"original claim": "The model finds the pages that will decline.",
     "rewritten (safe) claim":
        f"Measured: on whole clients it had not seen, model scores were associated with a page being marked 'down' "
        f"(Random Forest AUC {rf_after['AUC']:.3f}, Logistic Regression {lr_after['AUC']:.3f}; 0.5 would be chance; base rate {rf_after['base rate']:.2f}). "
        "This is decision-support for choosing which pages to review first, not a forecast of future decline."},
    {"original claim": "The model beats the baseline rule.",
     "rewritten (safe) claim":
        f"Directional: precision@50 was {rf_after['P@50']:.3f} for the Random Forest under client-grouped validation vs. a base rate of {rf_after['base rate']:.2f}. "
        "Compare this with the baseline rule's number on the same folds (Week-5 table); gaps of a few pages out of 50 are within the fold-to-fold spread."},
    {"original claim": "The random-split score shows how good the model is.",
     "rewritten (safe) claim":
        f"Measured: a random split by page gave AUC {rf_before['AUC']:.3f}; holding out whole clients gave {rf_after['AUC']:.3f}. "
        "The difference is memorisation of client habits, so I report the grouped number only."},
    {"original claim": "Low CTR leads to decline, so fixing titles will stop it.",
     "rewritten (safe) claim":
        "Observed: within a position band, pages with the lowest CTR were marked 'down' more often (Week-4 signal check). "
        "This is an association in one snapshot; it does not show that rewriting a title would reverse a decline."},
    {"original claim": "These features are all knowable before the outcome.",
     "rewritten (safe) claim":
        f"Partly: the 90-day features overlap the label window. A version using first-30-day features only scored AUC {rf_early['AUC (first-30-day features only)']:.3f} "
        f"vs. {rf_early['AUC (90-day features)']:.3f} for the 90-day version (Random Forest, grouped by client). The size of that gap tells how much of the signal came from the overlap."},
])
display(claims)

# Receipts: aggregate-only metrics file that is safe to commit
os.makedirs("work/outputs", exist_ok=True)
receipts = {"seed": SEED, "slice": "starter CSV gated slice (500+ impressions, position 1-20)",
            "before_after": before_after.to_dict(orient="records"), "deliberate_leak": leak_table.to_dict(orient="records"),
            "window_test": window_table.to_dict(orient="records"),
            "caveat": "Proxy label from the same 90-day window as most features; no dates, so no time-aware split; associations, not causes."}
with open("work/outputs/w06_audit_metrics.json", "w") as f:
    json.dump(receipts, f, indent=2, default=str)
print("Wrote work/outputs/w06_audit_metrics.json")

,original claim,rewritten (safe) claim
0,The model finds the pages that will decline.,"Measured: on whole clients it had not seen, model scores were associated with a page being marked 'down' (Random Forest AUC 0.616, Logistic Regression 0.608; 0.5 would be chance; base rate 0.61). This is decision-support for choosing which pages to review first, not a forecast of future decline."
1,The model beats the baseline rule.,Directional: precision@50 was 0.872 for the Random Forest under client-grouped validation vs. a base rate of 0.61. Compare this with the baseline rule's number on the same folds (Week-5 table); gaps of a few pages out of 50 are within the fold-to-fold spread.
2,The random-split score shows how good the model is.,"Measured: a random split by page gave AUC 0.709; holding out whole clients gave 0.616. The difference is memorisation of client habits, so I report the grouped number only."
3,"Low CTR leads to decline, so fixing titles will stop it.","Observed: within a position band, pages with the lowest CTR were marked 'down' more often (Week-4 signal check). This is an association in one snapshot; it does not show that rewriting a title would reverse a decline."
4,These features are all knowable before the outcome.,"Partly: the 90-day features overlap the label window. A version using first-30-day features only scored AUC 0.582 vs. 0.616 for the 90-day version (Random Forest, grouped by client). The size of that gap tells how much of the signal came from the overlap."


Wrote work/outputs/w06_audit_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.